# 05 — Compute Evaluation Metrics

Questo notebook calcola le metriche quantitative relative ai task
classificatori e al task STS-B-style.

Vengono valutate le seguenti configurazioni:

- modello base;
- adapter `regulation`;
- adapter `positive_legends`;
- adapter `negative_legends`;
- adapter `negative_to_correction`.

Il notebook produce:

1. metriche aggregate per ciascuna coppia modello-task;
2. precision, recall e F1 per ogni classe;
3. confusion matrix assolute e normalizzate;
4. risultati dettagliati per singolo esempio;
5. metriche specifiche per STS-B;
6. primary score associato a ciascun task;
7. dataset generativo pronto per il notebook 06.

Il notebook non assegna posizioni in classifica e non calcola un vincitore
complessivo. Ranking, confronti tra modelli e analisi degli scostamenti saranno
gestiti nei notebook successivi.

Tutte le predizioni utilizzate devono essere già state validate e normalizzate
dal notebook `04_parse_predictions.ipynb`.

In [4]:
from pathlib import Path
from collections import Counter
import json
import math
import warnings

import numpy as np
import pandas as pd

from scipy.stats import pearsonr, spearmanr

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

In [5]:
PROJECT_ROOT = Path(
    r"C:\Users\fede2\Desktop\Project"
).resolve()

INPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "predictions"
    / "hf_predictions_parsed.jsonl"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "metrics"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


MODEL_ORDER = [
    "base",
    "regulation",
    "positive_legends",
    "negative_legends",
    "negative_to_correction",
]


TASK_CONFIG = {
    "sst2_style": {
        "task_type": "classification",
        "labels": [
            "compliant",
            "non_compliant",
        ],
        "primary_metric_name": "accuracy",
    },

    "paraphrase_style": {
        "task_type": "classification",
        "labels": [
            "equivalent",
            "not_equivalent",
        ],
        "positive_label": "equivalent",
        "primary_metric_name": (
            "mean_accuracy_f1_equivalent"
        ),
    },

    "qqp_style": {
        "task_type": "classification",
        "labels": [
            "duplicate",
            "not_duplicate",
        ],
        "positive_label": "duplicate",
        "primary_metric_name": (
            "mean_accuracy_f1_duplicate"
        ),
    },

    "mnli_style": {
        "task_type": "classification",
        "labels": [
            "entailment",
            "contradiction",
            "neutral",
        ],
        "primary_metric_name": "accuracy",
    },

    "rte_style": {
        "task_type": "classification",
        "labels": [
            "entailment",
            "not_entailment",
        ],
        "primary_metric_name": "accuracy",
    },

    "bias_detection": {
        "task_type": "classification",
        "labels": [
            "biased",
            "unbiased",
        ],
        "positive_label": "biased",
        "primary_metric_name": "macro_f1",
    },

    "stsb_style": {
        "task_type": "ordinal",
        "labels": [
            0,
            1,
            2,
            3,
            4,
            5,
        ],
        "primary_metric_name": (
            "mean_pearson_spearman"
        ),
    },

    "grounded_answer_generation": {
        "task_type": "generation",
        "labels": None,
        "primary_metric_name": None,
    },
}


CLASSIFICATION_TASKS = [
    task_name
    for task_name, config in TASK_CONFIG.items()
    if config["task_type"] == "classification"
]

STSB_TASK = "stsb_style"

GENERATION_TASK = (
    "grounded_answer_generation"
)


EXPECTED_EXAMPLES_PER_MODEL_TASK = 20

EXPECTED_TOTAL_PREDICTIONS = (
    len(MODEL_ORDER)
    * len(TASK_CONFIG)
    * EXPECTED_EXAMPLES_PER_MODEL_TASK
)


print("Input:", INPUT_PATH)
print("Output:", OUTPUT_DIR)
print("Modelli attesi:", len(MODEL_ORDER))
print("Task attesi:", len(TASK_CONFIG))
print(
    "Predizioni complessive attese:",
    EXPECTED_TOTAL_PREDICTIONS,
)

Input: C:\Users\fede2\Desktop\Project\data\predictions\hf_predictions_parsed.jsonl
Output: C:\Users\fede2\Desktop\Project\data\metrics
Modelli attesi: 5
Task attesi: 8
Predizioni complessive attese: 800


In [6]:
def read_jsonl(path: Path) -> list[dict]:
    records = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as file:
        for line_number, line in enumerate(
            file,
            start=1,
        ):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)

            except json.JSONDecodeError as error:
                raise ValueError(
                    f"JSON non valido alla riga "
                    f"{line_number}: {error}"
                ) from error

            if not isinstance(record, dict):
                raise TypeError(
                    f"La riga {line_number} non contiene "
                    "un oggetto JSON."
                )

            records.append(record)

    return records


if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"File parsato non trovato:\n{INPUT_PATH}\n\n"
        "Eseguire prima il notebook 04."
    )


parsed_records = read_jsonl(
    INPUT_PATH
)

predictions_df = pd.DataFrame(
    parsed_records
)


print(
    "Predizioni caricate:",
    len(predictions_df),
)

display(
    predictions_df.head()
)

Predizioni caricate: 800


,example_id,task_name,model_name,adapter_name,y_true,allowed_labels,raw_output,generation_config,elapsed_seconds,prompt_sha256,...,status,error,y_pred,is_valid_prediction,parse_status,parse_method,matched_labels,normalized_output,parser_version,matched_scores
0,SST2_001,sst2_style,base,NaN,compliant,"[compliant, non_compliant]",non_compliant,"{'do_sample': False, 'max_new_tokens': 8}",0.9049,9ec757ccaff7d146fee148be4835da4d0a684dcc8f17a8...,...,completed,None,non_compliant,True,valid,exact_label,[non_compliant],non compliant,1.0,NaN
1,SST2_002,sst2_style,base,NaN,non_compliant,"[compliant, non_compliant]",non_compliant,"{'do_sample': False, 'max_new_tokens': 8}",0.3611,a7861d294789ddaf530b240050b219e9102c21d885d82d...,...,completed,None,non_compliant,True,valid,exact_label,[non_compliant],non compliant,1.0,NaN
2,SST2_003,sst2_style,base,NaN,compliant,"[compliant, non_compliant]",non_compliant,"{'do_sample': False, 'max_new_tokens': 8}",0.3560,bb0ce7ffdcad7dea29b782ba83803aa8d90ad6249ad37e...,...,completed,None,non_compliant,True,valid,exact_label,[non_compliant],non compliant,1.0,NaN
3,SST2_004,sst2_style,base,NaN,non_compliant,"[compliant, non_compliant]",non_compliant,"{'do_sample': False, 'max_new_tokens': 8}",0.3610,86e17f63a70ce575f58cfac4314764ce60e416a93d9551...,...,completed,None,non_compliant,True,valid,exact_label,[non_compliant],non compliant,1.0,NaN
4,SST2_005,sst2_style,base,NaN,compliant,"[compliant, non_compliant]",non_compliant,"{'do_sample': False, 'max_new_tokens': 8}",0.3600,7cb80ca866a9dd805c0563155de9132817ec816bd60aae...,...,completed,None,non_compliant,True,valid,exact_label,[non_compliant],non compliant,1.0,NaN


In [7]:
REQUIRED_COLUMNS = {
    "example_id",
    "task_name",
    "model_name",
    "y_true",
    "y_pred",
}


missing_columns = (
    REQUIRED_COLUMNS
    - set(predictions_df.columns)
)

if missing_columns:
    raise RuntimeError(
        "Nel file parsato mancano le colonne: "
        + ", ".join(
            sorted(missing_columns)
        )
    )


if "qnli_style" in set(
    predictions_df["task_name"]
):
    raise RuntimeError(
        "Il file contiene ancora il task qnli_style."
    )


unknown_tasks = (
    set(predictions_df["task_name"])
    - set(TASK_CONFIG)
)

if unknown_tasks:
    raise RuntimeError(
        "Task non configurati: "
        + ", ".join(
            sorted(unknown_tasks)
        )
    )


unknown_models = (
    set(predictions_df["model_name"])
    - set(MODEL_ORDER)
)

if unknown_models:
    raise RuntimeError(
        "Modelli non configurati: "
        + ", ".join(
            sorted(unknown_models)
        )
    )


missing_models = (
    set(MODEL_ORDER)
    - set(predictions_df["model_name"])
)

if missing_models:
    raise RuntimeError(
        "Modelli mancanti: "
        + ", ".join(
            sorted(missing_models)
        )
    )


duplicate_mask = predictions_df.duplicated(
    subset=[
        "model_name",
        "example_id",
    ],
    keep=False,
)

if duplicate_mask.any():
    display(
        predictions_df.loc[
            duplicate_mask,
            [
                "model_name",
                "task_name",
                "example_id",
            ],
        ]
    )

    raise RuntimeError(
        "Sono presenti coppie duplicate "
        "(model_name, example_id)."
    )


if "is_valid_prediction" in predictions_df.columns:
    invalid_mask = (
        predictions_df[
            "is_valid_prediction"
        ] != True
    )

    if invalid_mask.any():
        display(
            predictions_df.loc[
                invalid_mask,
                [
                    "model_name",
                    "task_name",
                    "example_id",
                    "raw_output",
                    "parse_status",
                ],
            ]
        )

        raise RuntimeError(
            "Sono presenti predizioni non valide."
        )


counts_table = (
    predictions_df
    .groupby(
        [
            "model_name",
            "task_name",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        index=MODEL_ORDER,
        columns=list(TASK_CONFIG),
        fill_value=0,
    )
)

display(counts_table)


incorrect_counts = []

for model_name in MODEL_ORDER:
    for task_name in TASK_CONFIG:
        actual_count = int(
            counts_table.loc[
                model_name,
                task_name,
            ]
        )

        if (
            actual_count
            != EXPECTED_EXAMPLES_PER_MODEL_TASK
        ):
            incorrect_counts.append(
                f"{model_name}/{task_name}: "
                f"{actual_count} invece di "
                f"{EXPECTED_EXAMPLES_PER_MODEL_TASK}"
            )


if incorrect_counts:
    raise RuntimeError(
        "Conteggi non validi:\n- "
        + "\n- ".join(incorrect_counts)
    )


example_consistency = (
    predictions_df
    .assign(
        y_true_as_string=(
            predictions_df["y_true"]
            .astype(str)
        )
    )
    .groupby("example_id")
    .agg(
        number_of_models=(
            "model_name",
            "nunique",
        ),
        number_of_tasks=(
            "task_name",
            "nunique",
        ),
        number_of_ground_truths=(
            "y_true_as_string",
            "nunique",
        ),
    )
)


inconsistent_examples = (
    example_consistency[
        (
            example_consistency[
                "number_of_models"
            ]
            != len(MODEL_ORDER)
        )
        |
        (
            example_consistency[
                "number_of_tasks"
            ]
            != 1
        )
        |
        (
            example_consistency[
                "number_of_ground_truths"
            ]
            != 1
        )
    ]
)

if not inconsistent_examples.empty:
    display(inconsistent_examples)

    raise RuntimeError(
        "La ground truth o il task non sono "
        "coerenti tra i modelli."
    )


if (
    len(predictions_df)
    != EXPECTED_TOTAL_PREDICTIONS
):
    raise RuntimeError(
        f"Predizioni trovate: {len(predictions_df)}. "
        f"Predizioni attese: "
        f"{EXPECTED_TOTAL_PREDICTIONS}."
    )


print("Validazione completata correttamente.")

task_name,sst2_style,paraphrase_style,qqp_style,mnli_style,rte_style,bias_detection,stsb_style,grounded_answer_generation
model_name,,,,,,,,
base,20,20,20,20,20,20,20,20
regulation,20,20,20,20,20,20,20,20
positive_legends,20,20,20,20,20,20,20,20
negative_legends,20,20,20,20,20,20,20,20
negative_to_correction,20,20,20,20,20,20,20,20


Validazione completata correttamente.


In [8]:
classification_df = predictions_df[
    predictions_df["task_name"].isin(
        CLASSIFICATION_TASKS
    )
].copy()


for task_name in CLASSIFICATION_TASKS:
    task_mask = (
        classification_df["task_name"]
        == task_name
    )

    classification_df.loc[
        task_mask,
        "y_true",
    ] = (
        classification_df.loc[
            task_mask,
            "y_true",
        ]
        .astype(str)
        .str.strip()
    )

    classification_df.loc[
        task_mask,
        "y_pred",
    ] = (
        classification_df.loc[
            task_mask,
            "y_pred",
        ]
        .astype(str)
        .str.strip()
    )

    expected_labels = set(
        TASK_CONFIG[
            task_name
        ]["labels"]
    )

    actual_true_labels = set(
        classification_df.loc[
            task_mask,
            "y_true",
        ]
    )

    actual_predicted_labels = set(
        classification_df.loc[
            task_mask,
            "y_pred",
        ]
    )

    invalid_true_labels = (
        actual_true_labels
        - expected_labels
    )

    invalid_predicted_labels = (
        actual_predicted_labels
        - expected_labels
    )

    if invalid_true_labels:
        raise RuntimeError(
            f"{task_name}: ground truth non valide: "
            f"{sorted(invalid_true_labels)}"
        )

    if invalid_predicted_labels:
        raise RuntimeError(
            f"{task_name}: predizioni non valide: "
            f"{sorted(invalid_predicted_labels)}"
        )


stsb_df = predictions_df[
    predictions_df["task_name"]
    == STSB_TASK
].copy()

stsb_df["y_true"] = pd.to_numeric(
    stsb_df["y_true"],
    errors="raise",
).astype(int)

stsb_df["y_pred"] = pd.to_numeric(
    stsb_df["y_pred"],
    errors="raise",
).astype(int)


valid_stsb_scores = set(
    TASK_CONFIG[
        STSB_TASK
    ]["labels"]
)

if not set(stsb_df["y_true"]).issubset(
    valid_stsb_scores
):
    raise RuntimeError(
        "STS-B contiene ground truth fuori "
        "dall'intervallo 0–5."
    )

if not set(stsb_df["y_pred"]).issubset(
    valid_stsb_scores
):
    raise RuntimeError(
        "STS-B contiene predizioni fuori "
        "dall'intervallo 0–5."
    )


generation_df = predictions_df[
    predictions_df["task_name"]
    == GENERATION_TASK
].copy()


print(
    "Predizioni classificatorie:",
    len(classification_df),
)

print(
    "Predizioni STS-B:",
    len(stsb_df),
)

print(
    "Predizioni generative:",
    len(generation_df),
)

Predizioni classificatorie: 600
Predizioni STS-B: 100
Predizioni generative: 100


## Metriche dei task classificatori

Per ciascuna coppia modello-task vengono calcolate:

- numero di esempi;
- numero di predizioni corrette;
- numero di errori;
- accuracy;
- macro-precision;
- macro-recall;
- macro-F1;
- precision, recall, F1 e support per ogni classe;
- distribuzione delle classi vere e predette;
- confusion matrix assoluta;
- confusion matrix normalizzata rispetto alla classe reale;
- tipi di errore osservati.

Le metriche primarie seguono l'impostazione GLUE quando esiste un task
di riferimento:

- SST-2-style: accuracy;
- MRPC-style: media tra accuracy e F1 della classe `equivalent`;
- QQP-style: media tra accuracy e F1 della classe `duplicate`;
- MNLI-style: accuracy;
- RTE-style: accuracy.

Per il task custom `bias_detection` viene utilizzata come metrica primaria
la macro-F1, mantenendo comunque disponibili accuracy e metriche per classe.

Le metriche primarie vengono salvate, ma non utilizzate in questo notebook
per costruire classifiche.

In [9]:
def compute_classification_metrics(
    group: pd.DataFrame,
    model_name: str,
    task_name: str,
):
    config = TASK_CONFIG[
        task_name
    ]

    labels = config[
        "labels"
    ]

    y_true = group[
        "y_true"
    ].tolist()

    y_pred = group[
        "y_pred"
    ].tolist()


    accuracy = float(
        accuracy_score(
            y_true,
            y_pred,
        )
    )


    (
        macro_precision,
        macro_recall,
        macro_f1,
        _,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=labels,
        average="macro",
        zero_division=0,
    )


    (
        class_precision,
        class_recall,
        class_f1,
        class_support,
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=labels,
        average=None,
        zero_division=0,
    )


    class_rows = []

    class_metric_map = {}

    for (
        label,
        precision,
        recall,
        f1,
        support,
    ) in zip(
        labels,
        class_precision,
        class_recall,
        class_f1,
        class_support,
    ):
        class_rows.append({
            "model_name": model_name,
            "task_name": task_name,
            "class_name": label,
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "support": int(support),
        })

        class_metric_map[
            label
        ] = {
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "support": int(support),
        }


    n_examples = len(group)

    n_correct = int(
        np.sum(
            np.asarray(y_true)
            == np.asarray(y_pred)
        )
    )

    n_errors = (
        n_examples
        - n_correct
    )


    primary_metric_name = config[
        "primary_metric_name"
    ]

    primary_score_formula = None

    secondary_metric_name = None

    secondary_metric_value = np.nan


    if task_name == "sst2_style":
        primary_score = accuracy

        primary_score_formula = (
            "accuracy"
        )


    elif task_name == "paraphrase_style":
        positive_f1 = class_metric_map[
            "equivalent"
        ]["f1"]

        primary_score = (
            accuracy
            + positive_f1
        ) / 2

        secondary_metric_name = (
            "f1_equivalent"
        )

        secondary_metric_value = (
            positive_f1
        )

        primary_score_formula = (
            "(accuracy + f1_equivalent) / 2"
        )


    elif task_name == "qqp_style":
        positive_f1 = class_metric_map[
            "duplicate"
        ]["f1"]

        primary_score = (
            accuracy
            + positive_f1
        ) / 2

        secondary_metric_name = (
            "f1_duplicate"
        )

        secondary_metric_value = (
            positive_f1
        )

        primary_score_formula = (
            "(accuracy + f1_duplicate) / 2"
        )


    elif task_name in {
        "mnli_style",
        "rte_style",
    }:
        primary_score = accuracy

        primary_score_formula = (
            "accuracy"
        )


    elif task_name == "bias_detection":
        primary_score = float(
            macro_f1
        )

        primary_score_formula = (
            "macro_f1"
        )


    else:
        raise ValueError(
            f"Configurazione metrica non definita "
            f"per {task_name}."
        )


    summary_row = {
        "model_name": model_name,
        "task_name": task_name,

        "n_examples": int(
            n_examples
        ),

        "n_correct": int(
            n_correct
        ),

        "n_errors": int(
            n_errors
        ),

        "accuracy": accuracy,

        "macro_precision": float(
            macro_precision
        ),

        "macro_recall": float(
            macro_recall
        ),

        "macro_f1": float(
            macro_f1
        ),

        "secondary_metric_name": (
            secondary_metric_name
        ),

        "secondary_metric_value": (
            secondary_metric_value
        ),

        "primary_metric_name": (
            primary_metric_name
        ),

        "primary_score": float(
            primary_score
        ),

        "primary_score_formula": (
            primary_score_formula
        ),
    }


    raw_matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=labels,
    )

    normalized_matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=labels,
        normalize="true",
    )


    matrix_rows = []

    for true_index, true_label in enumerate(
        labels
    ):
        for (
            predicted_index,
            predicted_label,
        ) in enumerate(labels):
            matrix_rows.append({
                "model_name": model_name,
                "task_name": task_name,
                "true_label": true_label,
                "predicted_label": (
                    predicted_label
                ),
                "count": int(
                    raw_matrix[
                        true_index,
                        predicted_index,
                    ]
                ),
                "normalized_by_true_label": float(
                    normalized_matrix[
                        true_index,
                        predicted_index,
                    ]
                ),
            })


    true_distribution = Counter(
        y_true
    )

    predicted_distribution = Counter(
        y_pred
    )

    distribution_rows = []

    for label in labels:
        distribution_rows.append({
            "model_name": model_name,
            "task_name": task_name,
            "class_name": label,
            "true_count": int(
                true_distribution.get(
                    label,
                    0,
                )
            ),
            "predicted_count": int(
                predicted_distribution.get(
                    label,
                    0,
                )
            ),
        })


    error_group = group[
        group["y_true"]
        != group["y_pred"]
    ]

    error_rows = []

    if not error_group.empty:
        error_counts = (
            error_group
            .groupby(
                [
                    "y_true",
                    "y_pred",
                ]
            )
            .size()
            .reset_index(
                name="count"
            )
        )

        for row in error_counts.itertuples(
            index=False
        ):
            error_rows.append({
                "model_name": model_name,
                "task_name": task_name,
                "true_label": row.y_true,
                "predicted_label": row.y_pred,
                "count": int(row.count),
            })


    example_rows = []

    for row in group.itertuples(
        index=False
    ):
        example_rows.append({
            "model_name": model_name,
            "task_name": task_name,
            "example_id": row.example_id,
            "y_true": row.y_true,
            "y_pred": row.y_pred,
            "is_correct": bool(
                row.y_true
                == row.y_pred
            ),
        })


    return {
        "summary": summary_row,
        "class_metrics": class_rows,
        "matrix_rows": matrix_rows,
        "raw_matrix": raw_matrix,
        "normalized_matrix": (
            normalized_matrix
        ),
        "labels": labels,
        "distribution": distribution_rows,
        "errors": error_rows,
        "examples": example_rows,
    }

In [10]:
classification_summary_rows = []

per_class_rows = []

confusion_matrix_rows = []

prediction_distribution_rows = []

classification_error_rows = []

classification_example_rows = []


confusion_matrices_raw = {}

confusion_matrices_normalized = {}


for model_name in MODEL_ORDER:
    confusion_matrices_raw[
        model_name
    ] = {}

    confusion_matrices_normalized[
        model_name
    ] = {}

    for task_name in CLASSIFICATION_TASKS:
        group = classification_df[
            (
                classification_df[
                    "model_name"
                ]
                == model_name
            )
            &
            (
                classification_df[
                    "task_name"
                ]
                == task_name
            )
        ].copy()

        group = group.sort_values(
            "example_id"
        )

        result = (
            compute_classification_metrics(
                group=group,
                model_name=model_name,
                task_name=task_name,
            )
        )

        classification_summary_rows.append(
            result["summary"]
        )

        per_class_rows.extend(
            result["class_metrics"]
        )

        confusion_matrix_rows.extend(
            result["matrix_rows"]
        )

        prediction_distribution_rows.extend(
            result["distribution"]
        )

        classification_error_rows.extend(
            result["errors"]
        )

        classification_example_rows.extend(
            result["examples"]
        )

        confusion_matrices_raw[
            model_name
        ][
            task_name
        ] = {
            "labels": result["labels"],
            "matrix": (
                result["raw_matrix"]
                .tolist()
            ),
        }

        confusion_matrices_normalized[
            model_name
        ][
            task_name
        ] = {
            "labels": result["labels"],
            "matrix": (
                result[
                    "normalized_matrix"
                ]
                .tolist()
            ),
        }


classification_metrics_df = pd.DataFrame(
    classification_summary_rows
)

per_class_metrics_df = pd.DataFrame(
    per_class_rows
)

confusion_matrices_long_df = pd.DataFrame(
    confusion_matrix_rows
)

prediction_distribution_df = pd.DataFrame(
    prediction_distribution_rows
)

classification_errors_df = pd.DataFrame(
    classification_error_rows
)

classification_results_df = pd.DataFrame(
    classification_example_rows
)


print(
    "Combinazioni modello-task analizzate:",
    len(classification_metrics_df),
)

Combinazioni modello-task analizzate: 30


In [11]:
classification_display_columns = [
    "model_name",
    "task_name",
    "n_correct",
    "n_errors",
    "accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "primary_metric_name",
    "primary_score",
]


display(
    classification_metrics_df[
        classification_display_columns
    ]
    .sort_values(
        [
            "task_name",
            "model_name",
        ]
    )
    .round(4)
)


display(
    per_class_metrics_df
    .sort_values(
        [
            "task_name",
            "model_name",
            "class_name",
        ]
    )
    .round(4)
)

,model_name,task_name,n_correct,n_errors,accuracy,macro_precision,macro_recall,macro_f1,primary_metric_name,primary_score
5,base,bias_detection,18,2,0.90,0.9167,0.9000,0.8990,macro_f1,0.8990
23,negative_legends,bias_detection,19,1,0.95,0.9545,0.9500,0.9499,macro_f1,0.9499
29,negative_to_correction,bias_detection,19,1,0.95,0.9545,0.9500,0.9499,macro_f1,0.9499
17,positive_legends,bias_detection,19,1,0.95,0.9545,0.9500,0.9499,macro_f1,0.9499
11,regulation,bias_detection,18,2,0.90,0.9000,0.9000,0.9000,macro_f1,0.9000
3,base,mnli_style,7,13,0.35,0.7593,0.4226,0.3023,accuracy,0.3500
21,negative_legends,mnli_style,9,11,0.45,0.7708,0.5179,0.4328,accuracy,0.4500
27,negative_to_correction,mnli_style,11,9,0.55,0.7857,0.6131,0.5273,accuracy,0.5500
15,positive_legends,mnli_style,9,11,0.45,0.7708,0.5179,0.4328,accuracy,0.4500
9,regulation,mnli_style,8,12,0.40,0.7647,0.4702,0.3737,accuracy,0.4000


,model_name,task_name,class_name,precision,recall,f1,support
11,base,bias_detection,biased,1.0000,0.8,0.8889,10
12,base,bias_detection,unbiased,0.8333,1.0,0.9091,10
50,negative_legends,bias_detection,biased,0.9091,1.0,0.9524,10
51,negative_legends,bias_detection,unbiased,1.0000,0.9,0.9474,10
63,negative_to_correction,bias_detection,biased,0.9091,1.0,0.9524,10
...,...,...,...,...,...,...,...
53,negative_to_correction,sst2_style,non_compliant,0.7692,1.0,0.8696,10
26,positive_legends,sst2_style,compliant,1.0000,0.6,0.7500,10
27,positive_legends,sst2_style,non_compliant,0.7143,1.0,0.8333,10
13,regulation,sst2_style,compliant,1.0000,0.2,0.3333,10


## Metriche STS-B-style

STS-B misura il grado di similarità semantica tra due testi mediante un punteggio
intero da 0 a 5.

Le metriche principali, coerenti con GLUE, sono:

- correlazione di Pearson;
- correlazione di Spearman;
- media tra Pearson e Spearman.

Vengono inoltre calcolate alcune metriche diagnostiche:

- MAE;
- RMSE;
- exact accuracy;
- accuracy entro ±1;
- accuracy entro ±2;
- mean signed error;
- distribuzione dell'errore assoluto;
- matrice tra punteggio reale e punteggio previsto.

Pearson e Spearman misurano l'andamento complessivo delle predizioni.
MAE e RMSE misurano invece la distanza numerica dai valori reali.

L'accuracy entro ±2 viene conservata come metrica aggiuntiva, ma non sarà
necessariamente utilizzata nelle tabelle principali o nei ranking.

In [12]:
def safe_pearson(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> float:
    if (
        len(np.unique(y_true)) < 2
        or len(np.unique(y_pred)) < 2
    ):
        return np.nan

    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore"
        )

        result = pearsonr(
            y_true,
            y_pred,
        )

    return float(
        result[0]
    )


def safe_spearman(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> float:
    if (
        len(np.unique(y_true)) < 2
        or len(np.unique(y_pred)) < 2
    ):
        return np.nan

    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore"
        )

        result = spearmanr(
            y_true,
            y_pred,
        )

    return float(
        result[0]
    )


def compute_stsb_metrics(
    group: pd.DataFrame,
    model_name: str,
):
    y_true = (
        group["y_true"]
        .to_numpy(
            dtype=int
        )
    )

    y_pred = (
        group["y_pred"]
        .to_numpy(
            dtype=int
        )
    )

    signed_error = (
        y_pred
        - y_true
    )

    absolute_error = np.abs(
        signed_error
    )


    pearson = safe_pearson(
        y_true,
        y_pred,
    )

    spearman = safe_spearman(
        y_true,
        y_pred,
    )


    if (
        np.isnan(pearson)
        or np.isnan(spearman)
    ):
        stsb_score = np.nan

    else:
        stsb_score = (
            pearson
            + spearman
        ) / 2


    mae = float(
        np.mean(
            absolute_error
        )
    )

    rmse = float(
        np.sqrt(
            np.mean(
                np.square(
                    signed_error
                )
            )
        )
    )

    exact_accuracy = float(
        np.mean(
            absolute_error == 0
        )
    )

    accuracy_within_1 = float(
        np.mean(
            absolute_error <= 1
        )
    )

    accuracy_within_2 = float(
        np.mean(
            absolute_error <= 2
        )
    )

    mean_signed_error = float(
        np.mean(
            signed_error
        )
    )


    summary = {
        "model_name": model_name,
        "task_name": STSB_TASK,
        "n_examples": int(
            len(group)
        ),
        "pearson": pearson,
        "spearman": spearman,
        "mae": mae,
        "rmse": rmse,
        "exact_accuracy": (
            exact_accuracy
        ),
        "accuracy_within_1": (
            accuracy_within_1
        ),
        "accuracy_within_2": (
            accuracy_within_2
        ),
        "mean_signed_error": (
            mean_signed_error
        ),
        "primary_metric_name": (
            "mean_pearson_spearman"
        ),
        "primary_score": (
            stsb_score
        ),
        "primary_score_formula": (
            "(pearson + spearman) / 2"
        ),
        "constant_prediction_flag": bool(
            len(np.unique(y_pred)) < 2
        ),
    }


    example_rows = []

    for (
        example_id,
        true_score,
        predicted_score,
        error,
        abs_error,
    ) in zip(
        group["example_id"],
        y_true,
        y_pred,
        signed_error,
        absolute_error,
    ):
        example_rows.append({
            "model_name": model_name,
            "task_name": STSB_TASK,
            "example_id": example_id,
            "y_true": int(
                true_score
            ),
            "y_pred": int(
                predicted_score
            ),
            "signed_error": int(
                error
            ),
            "absolute_error": int(
                abs_error
            ),
            "exact_match": bool(
                abs_error == 0
            ),
            "within_1": bool(
                abs_error <= 1
            ),
            "within_2": bool(
                abs_error <= 2
            ),
        })


    error_categories = {
        "exact_0": int(
            np.sum(
                absolute_error == 0
            )
        ),

        "off_by_1": int(
            np.sum(
                absolute_error == 1
            )
        ),

        "off_by_2": int(
            np.sum(
                absolute_error == 2
            )
        ),

        "off_by_3_or_more": int(
            np.sum(
                absolute_error >= 3
            )
        ),
    }


    error_distribution_rows = []

    for (
        category,
        count,
    ) in error_categories.items():
        error_distribution_rows.append({
            "model_name": model_name,
            "error_category": category,
            "count": count,
            "percentage": float(
                count
                / len(group)
            ),
        })


    score_labels = TASK_CONFIG[
        STSB_TASK
    ]["labels"]

    raw_matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=score_labels,
    )

    normalized_matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=score_labels,
        normalize="true",
    )


    return {
        "summary": summary,
        "examples": example_rows,
        "error_distribution": (
            error_distribution_rows
        ),
        "raw_matrix": raw_matrix,
        "normalized_matrix": (
            normalized_matrix
        ),
        "labels": score_labels,
    }

In [13]:
stsb_summary_rows = []

stsb_example_rows = []

stsb_error_distribution_rows = []


stsb_score_matrices_raw = {}

stsb_score_matrices_normalized = {}


for model_name in MODEL_ORDER:
    group = stsb_df[
        stsb_df["model_name"]
        == model_name
    ].copy()

    group = group.sort_values(
        "example_id"
    )

    result = compute_stsb_metrics(
        group=group,
        model_name=model_name,
    )

    stsb_summary_rows.append(
        result["summary"]
    )

    stsb_example_rows.extend(
        result["examples"]
    )

    stsb_error_distribution_rows.extend(
        result["error_distribution"]
    )

    stsb_score_matrices_raw[
        model_name
    ] = {
        "labels": result["labels"],
        "matrix": (
            result["raw_matrix"]
            .tolist()
        ),
    }

    stsb_score_matrices_normalized[
        model_name
    ] = {
        "labels": result["labels"],
        "matrix": (
            result[
                "normalized_matrix"
            ]
            .tolist()
        ),
    }


stsb_metrics_df = pd.DataFrame(
    stsb_summary_rows
)

stsb_results_df = pd.DataFrame(
    stsb_example_rows
)

stsb_error_distribution_df = pd.DataFrame(
    stsb_error_distribution_rows
)


display(
    stsb_metrics_df
    .sort_values(
        "model_name"
    )
    .round(4)
)


display(
    stsb_error_distribution_df
    .sort_values(
        [
            "model_name",
            "error_category",
        ]
    )
    .round(4)
)

,model_name,task_name,n_examples,pearson,spearman,mae,rmse,exact_accuracy,accuracy_within_1,accuracy_within_2,mean_signed_error,primary_metric_name,primary_score,primary_score_formula,constant_prediction_flag
0,base,stsb_style,20,0.8177,0.8409,1.05,1.3229,0.25,0.75,0.95,-0.85,mean_pearson_spearman,0.8293,(pearson + spearman) / 2,False
3,negative_legends,stsb_style,20,0.8595,0.8767,0.95,1.2450,0.30,0.80,0.95,-0.85,mean_pearson_spearman,0.8681,(pearson + spearman) / 2,False
4,negative_to_correction,stsb_style,20,0.8470,0.8694,1.10,1.3416,0.20,0.75,0.95,-1.00,mean_pearson_spearman,0.8582,(pearson + spearman) / 2,False
2,positive_legends,stsb_style,20,0.8329,0.8535,1.10,1.3784,0.25,0.70,0.95,-1.00,mean_pearson_spearman,0.8432,(pearson + spearman) / 2,False
1,regulation,stsb_style,20,0.8577,0.8748,1.00,1.2649,0.25,0.80,0.95,-0.90,mean_pearson_spearman,0.8663,(pearson + spearman) / 2,False


,model_name,error_category,count,percentage
0,base,exact_0,5,0.25
1,base,off_by_1,10,0.50
2,base,off_by_2,4,0.20
3,base,off_by_3_or_more,1,0.05
12,negative_legends,exact_0,6,0.30
13,negative_legends,off_by_1,10,0.50
14,negative_legends,off_by_2,3,0.15
15,negative_legends,off_by_3_or_more,1,0.05
16,negative_to_correction,exact_0,4,0.20
17,negative_to_correction,off_by_1,11,0.55


In [14]:
classification_primary_scores = (
    classification_metrics_df[
        [
            "model_name",
            "task_name",
            "primary_metric_name",
            "primary_score",
            "primary_score_formula",
        ]
    ]
    .copy()
)


stsb_primary_scores = (
    stsb_metrics_df[
        [
            "model_name",
            "task_name",
            "primary_metric_name",
            "primary_score",
            "primary_score_formula",
        ]
    ]
    .copy()
)


primary_task_scores_df = pd.concat(
    [
        classification_primary_scores,
        stsb_primary_scores,
    ],
    ignore_index=True,
)


primary_task_scores_df[
    "primary_score_percentage"
] = (
    primary_task_scores_df[
        "primary_score"
    ]
    * 100
)


display(
    primary_task_scores_df
    .sort_values(
        [
            "task_name",
            "model_name",
        ]
    )
    .round(4)
)

,model_name,task_name,primary_metric_name,primary_score,primary_score_formula,primary_score_percentage
5,base,bias_detection,macro_f1,0.8990,macro_f1,89.8990
23,negative_legends,bias_detection,macro_f1,0.9499,macro_f1,94.9875
29,negative_to_correction,bias_detection,macro_f1,0.9499,macro_f1,94.9875
17,positive_legends,bias_detection,macro_f1,0.9499,macro_f1,94.9875
11,regulation,bias_detection,macro_f1,0.9000,macro_f1,90.0000
3,base,mnli_style,accuracy,0.3500,accuracy,35.0000
21,negative_legends,mnli_style,accuracy,0.4500,accuracy,45.0000
27,negative_to_correction,mnli_style,accuracy,0.5500,accuracy,55.0000
15,positive_legends,mnli_style,accuracy,0.4500,accuracy,45.0000
9,regulation,mnli_style,accuracy,0.4000,accuracy,40.0000


## Preparazione del task generativo

Il task `grounded_answer_generation` non viene valutato quantitativamente
in questo notebook.

Questa sezione verifica soltanto che, per ogni modello:

- siano presenti tutti gli esempi attesi;
- la risposta di riferimento non sia vuota;
- la risposta generata non sia vuota;
- gli identificativi siano correttamente allineati.

Viene inoltre prodotto un file contenente:

- modello;
- identificativo dell'esempio;
- risposta di riferimento;
- risposta generata;
- lunghezza in parole.

Questo file sarà utilizzato direttamente dal notebook 06 per calcolare:

- BLEU;
- ROUGE;
- BERTScore;
- metriche di similarità;
- valutazione umana cieca.

In [15]:
def normalize_generation_text(
    value,
) -> str:
    if value is None:
        return ""

    return str(
        value
    ).strip()


generation_df[
    "reference_answer"
] = generation_df[
    "y_true"
].apply(
    normalize_generation_text
)

generation_df[
    "generated_answer"
] = generation_df[
    "y_pred"
].apply(
    normalize_generation_text
)


empty_references = generation_df[
    generation_df[
        "reference_answer"
    ]
    == ""
]

empty_predictions = generation_df[
    generation_df[
        "generated_answer"
    ]
    == ""
]


if not empty_references.empty:
    display(empty_references)

    raise RuntimeError(
        "Sono presenti reference generative vuote."
    )


if not empty_predictions.empty:
    display(empty_predictions)

    raise RuntimeError(
        "Sono presenti risposte generate vuote."
    )


generation_df[
    "reference_word_count"
] = (
    generation_df[
        "reference_answer"
    ]
    .str.split()
    .str.len()
)

generation_df[
    "generated_word_count"
] = (
    generation_df[
        "generated_answer"
    ]
    .str.split()
    .str.len()
)

generation_df[
    "word_length_ratio"
] = (
    generation_df[
        "generated_word_count"
    ]
    /
    generation_df[
        "reference_word_count"
    ]
)


generation_inputs_df = generation_df[
    [
        "model_name",
        "task_name",
        "example_id",
        "reference_answer",
        "generated_answer",
        "reference_word_count",
        "generated_word_count",
        "word_length_ratio",
    ]
].copy()


generation_readiness_df = (
    generation_inputs_df
    .groupby(
        "model_name"
    )
    .agg(
        n_examples=(
            "example_id",
            "count",
        ),
        average_reference_words=(
            "reference_word_count",
            "mean",
        ),
        average_generated_words=(
            "generated_word_count",
            "mean",
        ),
        median_generated_words=(
            "generated_word_count",
            "median",
        ),
        average_word_length_ratio=(
            "word_length_ratio",
            "mean",
        ),
    )
    .reset_index()
)


display(
    generation_readiness_df
    .round(4)
)

,model_name,n_examples,average_reference_words,average_generated_words,median_generated_words,average_word_length_ratio
0,base,20,24.0,48.65,45.0,2.0518
1,negative_legends,20,24.0,18.05,18.5,0.7628
2,negative_to_correction,20,24.0,19.90,21.0,0.8460
3,positive_legends,20,24.0,18.75,18.5,0.7937
4,regulation,20,24.0,22.60,21.0,0.9731


In [16]:
classification_export_df = (
    classification_metrics_df.copy()
)


stsb_export_df = (
    stsb_metrics_df.copy()
)


metrics_by_model_task_df = pd.concat(
    [
        classification_export_df,
        stsb_export_df,
    ],
    ignore_index=True,
    sort=False,
)


metrics_by_model_task_df = (
    metrics_by_model_task_df
    .sort_values(
        [
            "task_name",
            "model_name",
        ]
    )
    .reset_index(
        drop=True
    )
)


display(
    metrics_by_model_task_df
    .round(4)
)

,model_name,task_name,n_examples,n_correct,n_errors,accuracy,macro_precision,macro_recall,macro_f1,secondary_metric_name,...,primary_score_formula,pearson,spearman,mae,rmse,exact_accuracy,accuracy_within_1,accuracy_within_2,mean_signed_error,constant_prediction_flag
0,base,bias_detection,20,18.0,2.0,0.90,0.9167,0.9000,0.8990,NaN,...,macro_f1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,negative_legends,bias_detection,20,19.0,1.0,0.95,0.9545,0.9500,0.9499,NaN,...,macro_f1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,negative_to_correction,bias_detection,20,19.0,1.0,0.95,0.9545,0.9500,0.9499,NaN,...,macro_f1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,positive_legends,bias_detection,20,19.0,1.0,0.95,0.9545,0.9500,0.9499,NaN,...,macro_f1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,regulation,bias_detection,20,18.0,2.0,0.90,0.9000,0.9000,0.9000,NaN,...,macro_f1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,base,mnli_style,20,7.0,13.0,0.35,0.7593,0.4226,0.3023,NaN,...,accuracy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,negative_legends,mnli_style,20,9.0,11.0,0.45,0.7708,0.5179,0.4328,NaN,...,accuracy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,negative_to_correction,mnli_style,20,11.0,9.0,0.55,0.7857,0.6131,0.5273,NaN,...,accuracy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,positive_legends,mnli_style,20,9.0,11.0,0.45,0.7708,0.5179,0.4328,NaN,...,accuracy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,regulation,mnli_style,20,8.0,12.0,0.40,0.7647,0.4702,0.3737,NaN,...,accuracy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
OUTPUT_PATHS = {
    "metrics_by_model_task": (
        OUTPUT_DIR
        / "metrics_by_model_task.csv"
    ),

    "per_class_metrics": (
        OUTPUT_DIR
        / "per_class_metrics.csv"
    ),

    "classification_results_by_example": (
        OUTPUT_DIR
        / "classification_results_by_example.csv"
    ),

    "prediction_distribution": (
        OUTPUT_DIR
        / "prediction_distribution.csv"
    ),

    "classification_error_types": (
        OUTPUT_DIR
        / "classification_error_types.csv"
    ),

    "confusion_matrices_long": (
        OUTPUT_DIR
        / "confusion_matrices_long.csv"
    ),

    "confusion_matrices_raw": (
        OUTPUT_DIR
        / "confusion_matrices_raw.json"
    ),

    "confusion_matrices_normalized": (
        OUTPUT_DIR
        / "confusion_matrices_normalized.json"
    ),

    "stsb_metrics": (
        OUTPUT_DIR
        / "stsb_metrics.csv"
    ),

    "stsb_results_by_example": (
        OUTPUT_DIR
        / "stsb_results_by_example.csv"
    ),

    "stsb_error_distribution": (
        OUTPUT_DIR
        / "stsb_error_distribution.csv"
    ),

    "stsb_score_matrix_raw": (
        OUTPUT_DIR
        / "stsb_score_matrix_raw.json"
    ),

    "stsb_score_matrix_normalized": (
        OUTPUT_DIR
        / "stsb_score_matrix_normalized.json"
    ),

    "primary_task_scores": (
        OUTPUT_DIR
        / "primary_task_scores.csv"
    ),

    "generation_inputs_for_06": (
        OUTPUT_DIR
        / "generation_inputs_for_06.csv"
    ),

    "generation_readiness": (
        OUTPUT_DIR
        / "generation_readiness.csv"
    ),
}


metrics_by_model_task_df.to_csv(
    OUTPUT_PATHS[
        "metrics_by_model_task"
    ],
    index=False,
    encoding="utf-8",
)


per_class_metrics_df.to_csv(
    OUTPUT_PATHS[
        "per_class_metrics"
    ],
    index=False,
    encoding="utf-8",
)


classification_results_df.to_csv(
    OUTPUT_PATHS[
        "classification_results_by_example"
    ],
    index=False,
    encoding="utf-8",
)


prediction_distribution_df.to_csv(
    OUTPUT_PATHS[
        "prediction_distribution"
    ],
    index=False,
    encoding="utf-8",
)


classification_errors_df.to_csv(
    OUTPUT_PATHS[
        "classification_error_types"
    ],
    index=False,
    encoding="utf-8",
)


confusion_matrices_long_df.to_csv(
    OUTPUT_PATHS[
        "confusion_matrices_long"
    ],
    index=False,
    encoding="utf-8",
)


stsb_metrics_df.to_csv(
    OUTPUT_PATHS[
        "stsb_metrics"
    ],
    index=False,
    encoding="utf-8",
)


stsb_results_df.to_csv(
    OUTPUT_PATHS[
        "stsb_results_by_example"
    ],
    index=False,
    encoding="utf-8",
)


stsb_error_distribution_df.to_csv(
    OUTPUT_PATHS[
        "stsb_error_distribution"
    ],
    index=False,
    encoding="utf-8",
)


primary_task_scores_df.to_csv(
    OUTPUT_PATHS[
        "primary_task_scores"
    ],
    index=False,
    encoding="utf-8",
)


generation_inputs_df.to_csv(
    OUTPUT_PATHS[
        "generation_inputs_for_06"
    ],
    index=False,
    encoding="utf-8",
)


generation_readiness_df.to_csv(
    OUTPUT_PATHS[
        "generation_readiness"
    ],
    index=False,
    encoding="utf-8",
)


with OUTPUT_PATHS[
    "confusion_matrices_raw"
].open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        confusion_matrices_raw,
        file,
        ensure_ascii=False,
        indent=2,
    )


with OUTPUT_PATHS[
    "confusion_matrices_normalized"
].open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        confusion_matrices_normalized,
        file,
        ensure_ascii=False,
        indent=2,
    )


with OUTPUT_PATHS[
    "stsb_score_matrix_raw"
].open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        stsb_score_matrices_raw,
        file,
        ensure_ascii=False,
        indent=2,
    )


with OUTPUT_PATHS[
    "stsb_score_matrix_normalized"
].open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        stsb_score_matrices_normalized,
        file,
        ensure_ascii=False,
        indent=2,
    )


print("Output salvati:")

for output_name, output_path in OUTPUT_PATHS.items():
    print(
        f"- {output_name}: {output_path}"
    )

Output salvati:
- metrics_by_model_task: C:\Users\fede2\Desktop\Project\data\metrics\metrics_by_model_task.csv
- per_class_metrics: C:\Users\fede2\Desktop\Project\data\metrics\per_class_metrics.csv
- classification_results_by_example: C:\Users\fede2\Desktop\Project\data\metrics\classification_results_by_example.csv
- prediction_distribution: C:\Users\fede2\Desktop\Project\data\metrics\prediction_distribution.csv
- classification_error_types: C:\Users\fede2\Desktop\Project\data\metrics\classification_error_types.csv
- confusion_matrices_long: C:\Users\fede2\Desktop\Project\data\metrics\confusion_matrices_long.csv
- confusion_matrices_raw: C:\Users\fede2\Desktop\Project\data\metrics\confusion_matrices_raw.json
- confusion_matrices_normalized: C:\Users\fede2\Desktop\Project\data\metrics\confusion_matrices_normalized.json
- stsb_metrics: C:\Users\fede2\Desktop\Project\data\metrics\stsb_metrics.csv
- stsb_results_by_example: C:\Users\fede2\Desktop\Project\data\metrics\stsb_results_by_exampl

In [18]:
expected_classification_combinations = (
    len(MODEL_ORDER)
    * len(CLASSIFICATION_TASKS)
)

expected_primary_scores = (
    len(MODEL_ORDER)
    * (
        len(CLASSIFICATION_TASKS)
        + 1
    )
)


if (
    len(classification_metrics_df)
    != expected_classification_combinations
):
    raise RuntimeError(
        "Numero errato di combinazioni "
        "classificatorie."
    )


if (
    len(stsb_metrics_df)
    != len(MODEL_ORDER)
):
    raise RuntimeError(
        "Numero errato di risultati STS-B."
    )


if (
    len(primary_task_scores_df)
    != expected_primary_scores
):
    raise RuntimeError(
        "Numero errato di primary task score."
    )


if (
    len(classification_results_df)
    != len(classification_df)
):
    raise RuntimeError(
        "Il file classificatorio per esempio "
        "non contiene tutte le predizioni."
    )


if (
    len(stsb_results_df)
    != len(stsb_df)
):
    raise RuntimeError(
        "Il file STS-B per esempio "
        "non contiene tutte le predizioni."
    )


if (
    len(generation_inputs_df)
    != len(generation_df)
):
    raise RuntimeError(
        "Il file generativo per il notebook 06 "
        "non contiene tutte le risposte."
    )


missing_output_files = [
    str(path)
    for path in OUTPUT_PATHS.values()
    if not path.exists()
]

if missing_output_files:
    raise RuntimeError(
        "File di output mancanti:\n- "
        + "\n- ".join(
            missing_output_files
        )
    )


print("\n" + "=" * 90)
print("NOTEBOOK 05 COMPLETATO")
print("=" * 90)

print(
    "\nMetriche classificatorie:",
    len(classification_metrics_df),
)

print(
    "Metriche STS-B:",
    len(stsb_metrics_df),
)

print(
    "Primary task score:",
    len(primary_task_scores_df),
)

print(
    "Risultati classificatori per esempio:",
    len(classification_results_df),
)

print(
    "Risultati STS-B per esempio:",
    len(stsb_results_df),
)

print(
    "Risposte pronte per il notebook 06:",
    len(generation_inputs_df),
)

print("\nDirectory degli output:")
print(OUTPUT_DIR)

print(
    "\nIl notebook non ha calcolato "
    "ranking o confronti tra modelli."
)

print(
    "I file prodotti contengono tutte "
    "le informazioni necessarie per "
    "i notebook 06, 07 e 08."
)


NOTEBOOK 05 COMPLETATO

Metriche classificatorie: 30
Metriche STS-B: 5
Primary task score: 35
Risultati classificatori per esempio: 600
Risultati STS-B per esempio: 100
Risposte pronte per il notebook 06: 100

Directory degli output:
C:\Users\fede2\Desktop\Project\data\metrics

Il notebook non ha calcolato ranking o confronti tra modelli.
I file prodotti contengono tutte le informazioni necessarie per i notebook 06, 07 e 08.


data/metrics/
├── metrics_by_model_task.csv
├── per_class_metrics.csv
├── classification_results_by_example.csv
├── prediction_distribution.csv
├── classification_error_types.csv
├── confusion_matrices_long.csv
├── confusion_matrices_raw.json
├── confusion_matrices_normalized.json
├── stsb_metrics.csv
├── stsb_results_by_example.csv
├── stsb_error_distribution.csv
├── stsb_score_matrix_raw.json
├── stsb_score_matrix_normalized.json
├── primary_task_scores.csv
├── generation_inputs_for_06.csv
└── generation_readiness.csv